In [2]:
import class_ML_carbon as carb
import numpy as np
from netCDF4 import Dataset
from tensorflow import keras
from matplotlib import pyplot as plt

2025-10-23 11:39:40.518945: I tensorflow/core/platform/cpu_feature_guard.cc:210] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.
2025-10-23 11:39:41.826971: W tensorflow/compiler/tf2tensorrt/utils/py_utils.cc:38] TF-TRT Warning: Could not find TensorRT
/users/rsg/jos/miniconda3/lib/python3.9/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


# read in your data in a netCDF format. The data are in the form of datapoints x features. they are stored in two arrays: "inputs" and "outputs". This file has been saved for separate years and compressed due to GitHub requirements, it would need to be rebuild. 

In [3]:
inpfile = Dataset("Free_run_2016_data.nc")   

In [4]:
inputs = inpfile.variables["inputs"][:].transpose()   
outputs = inpfile.variables["outputs"][:].transpose()

# select the test subset from the data, we assume the test data are the last 20% of the data in the sequence, the first 80% were used for training and validation

In [5]:
inputs_train = inputs[:round(0.8*inputs.shape[0]),:]
outputs_train = outputs[:round(0.8*inputs.shape[0]),:]
inputs_test = inputs[round(0.8*inputs.shape[0]):,:]
outputs_test = outputs[round(0.8*inputs.shape[0]):,:]

# read the existing model architecture and weights

In [6]:
model = keras.models.load_model('./model')

ValueError: File format not supported: filepath=./model. Keras 3 only supports V3 `.keras` files and legacy H5 format files (`.h5` extension). Note that the legacy SavedModel format is not supported by `load_model()` in Keras 3. In order to reload a TensorFlow SavedModel as an inference-only layer in Keras 3, use `keras.layers.TFSMLayer(./model, call_endpoint='serving_default')` (note that your `call_endpoint` might have a different name).

In [6]:
model = keras.models.load_model('./model')

ValueError: File format not supported: filepath=./model. Keras 3 only supports V3 `.keras` files and legacy H5 format files (`.h5` extension). Note that the legacy SavedModel format is not supported by `load_model()` in Keras 3. In order to reload a TensorFlow SavedModel as an inference-only layer in Keras 3, use `keras.layers.TFSMLayer(./model, call_endpoint='serving_default')` (note that your `call_endpoint` might have a different name).

# use the model to predict the test data 

In [ ]:
pred_init = carb.ML_carbon(inputs = inputs_test, normalization_inputs=inputs_train, model = model)
pred = pred_init.predicted_values()

# compare the test data with the predicted data and plot them variable after variable (alternatively you can plot them in multi-panel plot.

In [ ]:
pred = carb.invert_normalization(data_norm = pred, data_ref = outputs_train)
outputs_list = pred_init.provide_output_names()

for i, output in enumerate(outputs_list):
    plt.figure(figsize=(8, 6))
    hb = plt.hexbin(pred[:,i], outputs_test[:,i], gridsize=50, cmap='plasma')
    plt.colorbar(hb, label='Point density')
    lims = [
        np.min([plt.xlim(), plt.ylim()]),  # min of both axes
        np.max([plt.xlim(), plt.ylim()]),  # max of both axes
    ]
    plt.plot(lims, lims, 'k--', alpha=0.75, zorder=0)  # black dashed line
    plt.xlim(lims)
    plt.ylim(lims)
    plt.xlabel('predicted')
    plt.ylabel('test data')
    plt.title(output)
    plt.show()